In [121]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import torch.nn as nn

In [122]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/master/data.csv')

In [123]:
df.head(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [124]:
df.drop(columns=['id' , 'Unnamed: 32'] , inplace=True)

In [125]:
X_train , x_test , y_train , y_test = train_test_split(df.iloc[:,1:] , df.iloc[:,0] , test_size=0.2)

In [126]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
x_test = scaler.transform(x_test)

In [127]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [128]:
X_train_tensor = torch.from_numpy(X_train)
x_test_tensor = torch.from_numpy(x_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [129]:
learning_rate = 0.1
epochs = 25

In [130]:
class NeuralNetwork(nn.Module):

  def __init__(self , X):

    super().__init__()

    self.linear = nn.Linear(X, 1)
    self.sigmoid = nn.Sigmoid()

    # self.weights = torch.rand(X.shape[1] , 1 ,dtype=torch.float64, requires_grad=True) # 30 rows 1 column
    # self.bias = torch.zeros(1 , dtype=torch.float64 , requires_grad=True)

  def forward(self , X):

    out = self.linear(X)
    out = self.sigmoid(out)
    # z = torch.matmul(X , self.weights) + self.bias
    # y_pred = torch.sigmoid(z)

    return out

  # def find_loss(self, y, y_pred):
  #   # clamp
  #     epsilon = 1e-7
  #     y_pred = torch.clamp(y_pred , epsilon , 1 - epsilon )  # clamp(input , min , max)
  #     loss = -(y_train_tensor * torch.log(y_pred) + 1 - y_train_tensor * torch.log(1 - y_pred)).mean()

  #     return loss




In [131]:

model = NeuralNetwork(X_train_tensor.shape[1])
model = model.double()


In [132]:
X_train_tensor

tensor([[-0.6268, -1.0931, -0.5661,  ..., -0.8284, -0.2748,  1.0518],
        [-0.2294,  0.2493, -0.1806,  ...,  0.5539,  1.3379,  1.0030],
        [-0.8745, -0.5733, -0.8671,  ..., -0.6122,  0.1571, -0.2915],
        ...,
        [-0.7362, -1.0263, -0.7436,  ..., -0.2732, -0.3751, -0.3305],
        [-0.3474, -0.7212, -0.3874,  ..., -0.8692,  1.1227, -0.7353],
        [-0.1660, -1.5557, -0.2488,  ..., -1.4251, -1.0078, -1.2067]],
       dtype=torch.float64)

In [133]:
X_train_tensor.shape

torch.Size([455, 30])

In [134]:
find_loss = nn.BCELoss()

In [135]:
# optimization now
optimizer = torch.optim.SGD(model.parameters() , lr=learning_rate)

In [136]:
# loop
for epoch in range(epochs):

# forward pass w.x + b
  y_pred = model(X_train_tensor)


# calc loss
  loss = find_loss(y_pred.float(),y_train_tensor.view(-1 , 1).float())

# we should clear the grads before backward()
  optimizer.zero_grad()

# backward pass
  loss.backward()

# parameters update
  # with torch.no_grad():
  #   model.linear.weight -= learning_rate * model.linear.weight.grad
  #   model.linear.bias -= learning_rate * model.linear.bias.grad

  optimizer.step()

    # zero grads

    # model.linear.weight.grad.zero_()
    # model.linear.bias.grad.zero_()

  # optimizer.zero_grad()

  print(f"Epoch : {epoch + 1} and loss : {loss.item()} ")

Epoch : 1 and loss : 0.6945816874504089 
Epoch : 2 and loss : 0.5334223508834839 
Epoch : 3 and loss : 0.44652891159057617 
Epoch : 4 and loss : 0.3925446569919586 
Epoch : 5 and loss : 0.35519328713417053 
Epoch : 6 and loss : 0.3274613916873932 
Epoch : 7 and loss : 0.3058560788631439 
Epoch : 8 and loss : 0.288430392742157 
Epoch : 9 and loss : 0.2740043103694916 
Epoch : 10 and loss : 0.26181599497795105 
Epoch : 11 and loss : 0.2513486444950104 
Epoch : 12 and loss : 0.24223780632019043 
Epoch : 13 and loss : 0.2342180758714676 
Epoch : 14 and loss : 0.22709102928638458 
Epoch : 15 and loss : 0.22070500254631042 
Epoch : 16 and loss : 0.21494178473949432 
Epoch : 17 and loss : 0.2097078263759613 
Epoch : 18 and loss : 0.20492790639400482 
Epoch : 19 and loss : 0.20054081082344055 
Epoch : 20 and loss : 0.19649626314640045 
Epoch : 21 and loss : 0.19275236129760742 
Epoch : 22 and loss : 0.18927407264709473 
Epoch : 23 and loss : 0.18603171408176422 
Epoch : 24 and loss : 0.1829999